# Gulf Renewable Energy Analysis
## Notebook 1: Data Cleaning & Preparation

**Project:** Targets vs. Reality — Mapping the Gulf's Renewable Energy Gap (2000–2023)  
**Countries:** Saudi Arabia | UAE | Oman | Kuwait  
**Author:** Dr. Adedamola Siyanbola  
**Tools:** Python, Pandas, NumPy  

---

### Datasets Used
| File | Indicator Code | Description |
|------|---------------|-------------|
| `API_EG.ELC.RNEW.ZS_DS2_en_csv_v2.csv` | EG.ELC.RNEW.ZS | Renewable electricity output (% of total) |
| `API_EN.ATM.CO2E.PC_DS2_en_csv_v2.csv` | EN.ATM.CO2E.PC | CO₂ emissions per capita (metric tonnes) |
| `API_NY.GDP.MKTP.CD_DS2_en_csv_v2.csv` | NY.GDP.MKTP.CD | GDP (current USD) |
| `API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2.csv` | EG.USE.PCAP.KG.OE | Energy use per capita (kg oil equivalent) |

> **Data source:** World Bank Open Data — data.worldbank.org

---
## Section 1: Import Libraries

In [1]:
# ── Standard Libraries ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

# ── Display Settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.3f}'.format)

print('✅ Libraries loaded successfully')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')

✅ Libraries loaded successfully
   Pandas  : 3.0.2
   NumPy   : 2.4.4


---
## Section 2: Configure File Paths & Constants

In [2]:
# ── Directory Paths ───────────────────────────────────────────────────────────
RAW_DIR     = '../data/raw/'
CLEANED_DIR = '../data/cleaned/'
os.makedirs(CLEANED_DIR, exist_ok=True)

# ── Raw File Names ────────────────────────────────────────────────────────────
# These match exactly the files in your data/raw/ folder
FILE_RENEWABLE  = 'API_EG.ELC.RNEW.ZS_DS2_en_csv_v2.csv'      # Renewable electricity %
FILE_CO2        = 'API_EN.ATM.CO2E.PC_DS2_en_csv_v2.csv'       # CO2 per capita
FILE_GDP        = 'API_NY.GDP.MKTP.CD_DS2_en_csv_v2.csv'       # GDP current USD
FILE_ENERGY     = 'API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2.csv'   # Energy use per capita

# ── Countries of Interest ─────────────────────────────────────────────────────
# These must match exactly how World Bank names them
COUNTRIES = ['Saudi Arabia', 'United Arab Emirates', 'Oman', 'Kuwait']

# ── Year Range ────────────────────────────────────────────────────────────────
START_YEAR = 2000
END_YEAR   = 2023   # Extended from Nigeria project to capture latest available data

print('✅ Configuration set')
print(f'   Countries  : {COUNTRIES}')
print(f'   Year range : {START_YEAR} – {END_YEAR}')
print(f'   Raw folder : {RAW_DIR}')

✅ Configuration set
   Countries  : ['Saudi Arabia', 'United Arab Emirates', 'Oman', 'Kuwait']
   Year range : 2000 – 2023
   Raw folder : ../data/raw/


---
## Section 3: Diagnostic — Inspect Raw Files Before Loading

Before we load anything, we look inside one file to understand its exact structure.
This prevents errors before they happen — a professional data analyst always inspects first.

In [3]:
# ── Quick diagnostic on the renewable electricity file ────────────────────────
print('=== RAW FILE DIAGNOSTIC ===')
print()

# Read the first 8 rows with no processing
df_diag = pd.read_csv(RAW_DIR + FILE_RENEWABLE, header=None, nrows=8)
print('First 8 rows of raw file (no processing):')
print(df_diag.to_string())
print()

# Also check the column names after skipping row 0
df_diag2 = pd.read_csv(RAW_DIR + FILE_RENEWABLE, skiprows=0)
print('Column names found:')
print(df_diag2.columns.tolist()[:8])
print()
print('First 3 values in Column 0:')
print(df_diag2.iloc[:3, 0].tolist())

=== RAW FILE DIAGNOSTIC ===

First 8 rows of raw file (no processing):
                                                             0                     1              2              3              4              5              6              7              8              9              10             11             12             13             14             15             16             17             18             19             20             21             22             23             24             25             26             27
0                                                   Series Name          Country Name  2000 [YR2000]  2001 [YR2001]  2002 [YR2002]  2003 [YR2003]  2004 [YR2004]  2005 [YR2005]  2006 [YR2006]  2007 [YR2007]  2008 [YR2008]  2009 [YR2009]  2010 [YR2010]  2011 [YR2011]  2012 [YR2012]  2013 [YR2013]  2014 [YR2014]  2015 [YR2015]  2016 [YR2016]  2017 [YR2017]  2018 [YR2018]  2019 [YR2019]  2020 [YR2020]  2021 [YR2021]  2022 [YR2022]  2023 [YR2023]  2024 [

---
## Section 4: World Bank Loader Function

This is the same smart loader we built for the Nigeria project — it automatically
detects the correct header row regardless of how many metadata lines the file has.
It also handles the World Bank's year column format: `'2000 [YR2000]'`.

In [4]:
def load_worldbank_csv(filepath, value_name):
    """
    Loads a World Bank indicator CSV and reshapes it from wide to long format.
    Handles variations in column naming including spacing differences.
    """

    # Step 1: Read raw file
    df_raw = pd.read_csv(filepath)
    df_raw.columns = df_raw.columns.str.strip()

    # Step 2: If 'Country Name' not found, scan for correct header row
    if 'Country Name' not in df_raw.columns:
        df_probe = pd.read_csv(filepath, header=None, nrows=10)
        header_row = None
        for i, row in df_probe.iterrows():
            if row.astype(str).str.contains('Country Name', case=False).any():
                header_row = i
                break
        if header_row is None:
            raise ValueError(f"Cannot find 'Country Name' in: {filepath}")
        df_raw = pd.read_csv(filepath, skiprows=header_row)
        df_raw.columns = df_raw.columns.str.strip()

    # Step 3: Auto-detect the exact column names for Country Name and Country Code
    # This handles spacing variations between World Bank file versions
    col_country_name = None
    col_country_code = None
    for col in df_raw.columns:
        col_clean = str(col).strip().lower()
        if col_clean == 'country name':
            col_country_name = col
        if col_clean == 'country code':
            col_country_code = col

    if col_country_name is None:
        raise ValueError(f"Cannot find 'Country Name' column in: {filepath}")

    # Step 4: Build id_cols — use Country Code only if it exists
    if col_country_code is not None:
        id_cols = [col_country_name, col_country_code]
    else:
        id_cols = [col_country_name]
        print(f'  ⚠️  No Country Code column found — using Country Name only')

    # Step 5: Identify year columns
    all_cols = df_raw.columns.tolist()
    year_col_map = {}
    for col in all_cols:
        for y in range(START_YEAR, END_YEAR + 1):
            if str(col).strip().startswith(str(y)):
                year_col_map[str(y)] = col
                break

    print(f'  📅 {len(year_col_map)} year columns found in: {os.path.basename(filepath)}')

    # Step 6: Select id columns and year columns
    year_cols_actual = [year_col_map[y] for y in sorted(year_col_map.keys())]
    df = df_raw[id_cols + year_cols_actual].copy()

    # Step 7: Filter to our 4 Gulf countries
    df = df[df[col_country_name].isin(COUNTRIES)].copy()

    # Step 8: Rename year columns to clean integers
    rename_map = {v: k for k, v in year_col_map.items()}
    df.rename(columns=rename_map, inplace=True)

    # Step 9: Drop rows with no country name
    df.dropna(subset=[col_country_name], inplace=True)

    # Step 10: Reshape wide to long
    year_cols_clean = sorted(year_col_map.keys())
    df = df.melt(
        id_vars    = id_cols,
        value_vars = year_cols_clean,
        var_name   = 'year',
        value_name = value_name
    )

    # Step 11: Standardise column names
    df.rename(columns={col_country_name: 'country_name'}, inplace=True)
    if col_country_code is not None:
        df.rename(columns={col_country_code: 'country_code'}, inplace=True)
    else:
        df['country_code'] = df['country_name'].map({
            'Saudi Arabia'         : 'SAU',
            'United Arab Emirates' : 'ARE',
            'Oman'                 : 'OMN',
            'Kuwait'               : 'KWT'
        })

    df['year'] = df['year'].astype(int)
    df.sort_values(['country_name', 'year'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df


print('✅ Loader function updated — handles column name variations')

✅ Loader function updated — handles column name variations


---
## Section 5: Load All 4 Datasets

In [5]:
print('Loading Gulf datasets...\n')

df_renewable = load_worldbank_csv(RAW_DIR + FILE_RENEWABLE, 'renewable_pct')
print(f'  ✅ Renewable Electricity  — {df_renewable.shape[0]} rows, {df_renewable.shape[1]} cols')

df_co2       = load_worldbank_csv(RAW_DIR + FILE_CO2,       'co2_per_capita')
print(f'  ✅ CO₂ Per Capita         — {df_co2.shape[0]} rows, {df_co2.shape[1]} cols')

df_gdp       = load_worldbank_csv(RAW_DIR + FILE_GDP,       'gdp_usd')
print(f'  ✅ GDP (Current USD)      — {df_gdp.shape[0]} rows, {df_gdp.shape[1]} cols')

df_energy    = load_worldbank_csv(RAW_DIR + FILE_ENERGY,    'energy_use_per_capita')
print(f'  ✅ Energy Use Per Capita  — {df_energy.shape[0]} rows, {df_energy.shape[1]} cols')

print()

# ── Confirm countries loaded correctly ────────────────────────────────────────
print('Countries in dataset:')
print(df_renewable['country_name'].unique().tolist())

Loading Gulf datasets...

  ⚠️  No Country Code column found — using Country Name only
  📅 24 year columns found in: API_EG.ELC.RNEW.ZS_DS2_en_csv_v2.csv
  ✅ Renewable Electricity  — 96 rows, 4 cols
  ⚠️  No Country Code column found — using Country Name only
  📅 24 year columns found in: API_EN.ATM.CO2E.PC_DS2_en_csv_v2.csv
  ✅ CO₂ Per Capita         — 96 rows, 4 cols
  ⚠️  No Country Code column found — using Country Name only
  📅 24 year columns found in: API_NY.GDP.MKTP.CD_DS2_en_csv_v2.csv
  ✅ GDP (Current USD)      — 96 rows, 4 cols
  ⚠️  No Country Code column found — using Country Name only
  📅 24 year columns found in: API_EG.USE.PCAP.KG.OE_DS2_en_csv_v2.csv
  ✅ Energy Use Per Capita  — 96 rows, 4 cols

Countries in dataset:
['Kuwait', 'Oman', 'Saudi Arabia', 'United Arab Emirates']


---
## Section 6: Inspect Each Dataset

In [6]:
def inspect_df(df, label):
    """Prints a clean summary: shape, year range, countries, types, missing values."""
    value_col = [c for c in df.columns if c not in ['country_name', 'country_code', 'year']][0]
    print('=' * 60)
    print(f'DATASET: {label}')
    print('=' * 60)
    print(f'  Shape          : {df.shape[0]} rows × {df.shape[1]} columns')
    print(f'  Year range     : {df["year"].min()} – {df["year"].max()}')
    print(f'  Countries      : {df["country_name"].unique().tolist()}')
    print()
    print('  Missing Values per Country:')
    missing = df.groupby('country_name')[value_col].apply(lambda x: x.isnull().sum())
    total   = END_YEAR - START_YEAR + 1
    for country, count in missing.items():
        pct = count / total * 100
        flag = '⚠️ ' if count > 0 else '✅'
        print(f'    {flag} {country:25s}  {count} missing ({pct:.0f}%)')
    print()
    print('  Sample (first 5 rows):')
    print(df.head())
    print()


inspect_df(df_renewable, 'Renewable Electricity (%)')
inspect_df(df_co2,       'CO₂ Per Capita (metric tonnes)')
inspect_df(df_gdp,       'GDP (Current USD)')
inspect_df(df_energy,    'Energy Use Per Capita')

DATASET: Renewable Electricity (%)
  Shape          : 96 rows × 4 columns
  Year range     : 2000 – 2023
  Countries      : ['Kuwait', 'Oman', 'Saudi Arabia', 'United Arab Emirates']

  Missing Values per Country:
    ✅ Kuwait                     0 missing (0%)
    ✅ Oman                       0 missing (0%)
    ✅ Saudi Arabia               0 missing (0%)
    ✅ United Arab Emirates       0 missing (0%)

  Sample (first 5 rows):
  country_name  year renewable_pct country_code
0       Kuwait  2000         0.000          KWT
1       Kuwait  2001         0.000          KWT
2       Kuwait  2002         0.000          KWT
3       Kuwait  2003         0.000          KWT
4       Kuwait  2004         0.000          KWT

DATASET: CO₂ Per Capita (metric tonnes)
  Shape          : 96 rows × 4 columns
  Year range     : 2000 – 2023
  Countries      : ['Kuwait', 'Oman', 'Saudi Arabia', 'United Arab Emirates']

  Missing Values per Country:
    ✅ Kuwait                     0 missing (0%)
    ✅ Oman  

---
## Section 7: Handle Missing Values

**Gulf data context:** Unlike the Nigeria project which had zero missing values,
Gulf energy data often has gaps — especially for Kuwait (renewable data is sparse)
and for energy use per capita in recent years.
We document every decision transparently.

In [8]:
def fill_missing_values(df, value_col):
    """
    Fills missing values per country using linear interpolation.
    Includes numeric conversion step to handle World Bank's
    object dtype columns before interpolating.
    """
    df = df.copy()

    # Step 1: Force convert to numeric — blank strings become NaN
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')

    # Step 2: Interpolate per country group
    df[value_col] = (
        df.groupby('country_name')[value_col]
          .transform(lambda x: x.interpolate(method='linear').ffill().bfill())
    )

    remaining = df[value_col].isnull().sum()
    if remaining > 0:
        print(f'  ⚠️  {value_col}: {remaining} values still missing — review manually')
    else:
        print(f'  ✅ {value_col}: all missing values resolved')

    return df


print('Filling missing values...\n')
df_renewable = fill_missing_values(df_renewable, 'renewable_pct')
df_co2       = fill_missing_values(df_co2,       'co2_per_capita')
df_gdp       = fill_missing_values(df_gdp,       'gdp_usd')
df_energy    = fill_missing_values(df_energy,    'energy_use_per_capita')

Filling missing values...

  ✅ renewable_pct: all missing values resolved
  ✅ co2_per_capita: all missing values resolved
  ✅ gdp_usd: all missing values resolved
  ✅ energy_use_per_capita: all missing values resolved


---
## Section 8: Validate Data Ranges

Gulf-specific validation rules:
- Renewable % must be 0–100
- CO₂ per capita for Gulf countries should be between 5–60 tonnes (confirmed in revalidation)
- GDP must be positive
- Energy use per capita must be positive

In [9]:
def validate_range(df, col, min_val=None, max_val=None, label=''):
    """Flags values outside expected bounds."""
    issues = pd.Series([False] * len(df), index=df.index)
    if min_val is not None:
        issues = issues | (df[col] < min_val)
    if max_val is not None:
        issues = issues | (df[col] > max_val)
    if issues.any():
        print(f'  ⚠️  {label}: {issues.sum()} value(s) outside [{min_val}, {max_val}]')
        print(df[issues][['country_name', 'year', col]])
    else:
        print(f'  ✅ {label}: all values within expected range')


print('Validating data ranges...\n')
validate_range(df_renewable, 'renewable_pct',       min_val=0,   max_val=100, label='Renewable %')
validate_range(df_co2,       'co2_per_capita',      min_val=0,   max_val=60,  label='CO₂ Per Capita')
validate_range(df_gdp,       'gdp_usd',             min_val=0,               label='GDP USD')
validate_range(df_energy,    'energy_use_per_capita', min_val=0,              label='Energy Use')

Validating data ranges...

  ✅ Renewable %: all values within expected range
  ✅ CO₂ Per Capita: all values within expected range
  ✅ GDP USD: all values within expected range
  ✅ Energy Use: all values within expected range


---
## Section 9: Add Derived Columns

These calculated columns will power the analysis and visualisations in Notebook 02.

In [10]:
# ── GDP in Billions (easier to read on charts) ────────────────────────────────
df_gdp['gdp_billion_usd'] = df_gdp['gdp_usd'] / 1e9

# ── Year-on-Year GDP Growth Rate ──────────────────────────────────────────────
df_gdp['gdp_growth_pct'] = (
    df_gdp.groupby('country_name')['gdp_usd']
          .pct_change() * 100
)

# ── Renewable Gap: distance from national 2030 targets ───────────────────────
# National 2030 renewable electricity targets (from national vision documents)
# Saudi Arabia: 50% by 2030 (Vision 2030 / NEOM targets)
# UAE: 44% by 2050 — using 32% as 2030 interim target (UAE Energy Strategy 2050)
# Oman: 20% by 2030 (Oman Vision 2040)
# Kuwait: 15% by 2030 (Kuwait Vision 2035)
TARGET_2030 = {
    'Saudi Arabia'          : 50.0,
    'United Arab Emirates'  : 32.0,
    'Oman'                  : 20.0,
    'Kuwait'                : 15.0,
}

df_renewable['target_2030'] = df_renewable['country_name'].map(TARGET_2030)
df_renewable['gap_to_target'] = df_renewable['target_2030'] - df_renewable['renewable_pct']
df_renewable['gap_to_target'] = df_renewable['gap_to_target'].clip(lower=0)  # no negative gaps
df_renewable['pct_of_target_achieved'] = (
    df_renewable['renewable_pct'] / df_renewable['target_2030'] * 100
).round(1)

# ── CO₂ Intensity of GDP (kg CO₂ per USD of GDP) ─────────────────────────────
# This requires merging CO₂ and GDP temporarily
df_co2_gdp = df_co2.merge(
    df_gdp[['country_name', 'year', 'gdp_usd']],
    on=['country_name', 'year'], how='left'
)

# ── Flag: Saudi Arabia (highlight in comparison charts) ───────────────────────
for df in [df_renewable, df_co2, df_gdp, df_energy]:
    df['is_saudi'] = df['country_name'] == 'Saudi Arabia'

print('✅ Derived columns added:')
print('   gdp_billion_usd          — GDP in billions (df_gdp)')
print('   gdp_growth_pct           — Year-on-year GDP growth (df_gdp)')
print('   target_2030              — National 2030 renewable target (df_renewable)')
print('   gap_to_target            — Distance from 2030 target (df_renewable)')
print('   pct_of_target_achieved   — % of 2030 target reached (df_renewable)')
print('   is_saudi                 — Boolean flag for Saudi Arabia rows')

print()
print('National 2030 renewable targets loaded:')
for country, target in TARGET_2030.items():
    print(f'   {country:30s}  {target}%')

✅ Derived columns added:
   gdp_billion_usd          — GDP in billions (df_gdp)
   gdp_growth_pct           — Year-on-year GDP growth (df_gdp)
   target_2030              — National 2030 renewable target (df_renewable)
   gap_to_target            — Distance from 2030 target (df_renewable)
   pct_of_target_achieved   — % of 2030 target reached (df_renewable)
   is_saudi                 — Boolean flag for Saudi Arabia rows

National 2030 renewable targets loaded:
   Saudi Arabia                    50.0%
   United Arab Emirates            32.0%
   Oman                            20.0%
   Kuwait                          15.0%


---
## Section 10: Merge All Datasets Into Master Table

In [11]:
merge_keys = ['country_name', 'country_code', 'year']

df_master = (
    df_renewable[merge_keys + ['renewable_pct', 'target_2030', 'gap_to_target',
                               'pct_of_target_achieved', 'is_saudi']]
    .merge(df_co2[merge_keys + ['co2_per_capita']],
           on=merge_keys, how='outer')
    .merge(df_gdp[merge_keys + ['gdp_usd', 'gdp_billion_usd', 'gdp_growth_pct']],
           on=merge_keys, how='outer')
    .merge(df_energy[merge_keys + ['energy_use_per_capita']],
           on=merge_keys, how='outer')
)

df_master.sort_values(['country_name', 'year'], inplace=True)
df_master.reset_index(drop=True, inplace=True)

print(f'✅ Master dataset created')
print(f'   Shape      : {df_master.shape[0]} rows × {df_master.shape[1]} columns')
print(f'   Countries  : {df_master["country_name"].unique().tolist()}')
print(f'   Year span  : {df_master["year"].min()} – {df_master["year"].max()}')
print()
print('All columns in master dataset:')
for col in df_master.columns:
    print(f'   {col}')
print()
df_master.head(8)

✅ Master dataset created
   Shape      : 96 rows × 13 columns
   Countries  : ['Kuwait', 'Oman', 'Saudi Arabia', 'United Arab Emirates']
   Year span  : 2000 – 2023

All columns in master dataset:
   country_name
   country_code
   year
   renewable_pct
   target_2030
   gap_to_target
   pct_of_target_achieved
   is_saudi
   co2_per_capita
   gdp_usd
   gdp_billion_usd
   gdp_growth_pct
   energy_use_per_capita



,country_name,country_code,year,renewable_pct,target_2030,gap_to_target,pct_of_target_achieved,is_saudi,co2_per_capita,gdp_usd,gdp_billion_usd,gdp_growth_pct,energy_use_per_capita
0,Kuwait,KWT,2000,0.000,15.000,15.000,0.000,False,28.660,"37,718,743,480.000",37.719,NaN,"9,498.024"
1,Kuwait,KWT,2001,0.000,15.000,15.000,0.000,False,29.318,"34,889,559,870.000",34.890,-7.501,"9,864.907"
2,Kuwait,KWT,2002,0.000,15.000,15.000,0.000,False,29.895,"38,135,788,414.000",38.136,9.304,"9,714.890"
3,Kuwait,KWT,2003,0.000,15.000,15.000,0.000,False,30.769,"47,874,582,232.000",47.875,25.537,"10,164.179"
4,Kuwait,KWT,2004,0.000,15.000,15.000,0.000,False,31.976,"59,439,090,601.000",59.439,24.156,"10,971.464"
5,Kuwait,KWT,2005,0.000,15.000,15.000,0.000,False,34.529,"80,798,630,137.000",80.799,35.935,"12,019.625"
6,Kuwait,KWT,2006,0.000,15.000,15.000,0.000,False,33.802,"101,557,000,000.000",101.557,25.691,"10,452.804"
7,Kuwait,KWT,2007,0.000,15.000,15.000,0.000,False,31.191,"114,634,000,000.000",114.634,12.877,"9,524.114"


---
## Section 11: Final Quality Report

In [13]:
print('FINAL QUALITY REPORT — GULF RENEWABLE ENERGY DATASET')
print('=' * 65)
print(f'  Total rows          : {len(df_master)}')
print(f'  Total columns       : {df_master.shape[1]}')
print(f'  Countries           : {df_master["country_name"].nunique()}')
print(f'  Year span           : {df_master["year"].min()} – {df_master["year"].max()}')
print()
print('  Missing Values in Master Dataset:')
print(df_master.isnull().sum().to_string())
print()

# ── Saudi Arabia summary ──────────────────────────────────────────────────────
sa = df_master[df_master['country_name'] == 'Saudi Arabia']
print('  Saudi Arabia — Key Statistics:')
print(f'  Renewable % in 2000  : {sa[sa["year"]==2000]["renewable_pct"].values[0]:.2f}%')

# Most recent year with data
sa_recent = sa[sa['renewable_pct'].notna()]
if not sa_recent.empty:
    latest_year = sa_recent['year'].max()
    latest_val  = sa_recent[sa_recent['year']==latest_year]['renewable_pct'].values[0]
    latest_gap  = sa_recent[sa_recent['year']==latest_year]['gap_to_target'].values[0]
    print(f'  Renewable % in {latest_year}  : {latest_val:.2f}%')
    print(f'  Gap to 2030 target   : {latest_gap:.1f} percentage points')
    print(f'  2030 target          : {TARGET_2030["Saudi Arabia"]}%')
print()

# ── All countries — latest renewable % vs target ──────────────────────────────
print('  Latest Renewable % vs. 2030 Target (all countries):')
print(f'  {"Country":30s} {"Latest %":>10s} {"Target":>10s} {"Gap":>10s} {"% Achieved":>12s}')
print('  ' + '-' * 75)
for country in COUNTRIES:
    df_c = df_master[(df_master['country_name']==country) & (df_master['renewable_pct'].notna())]
    if not df_c.empty:
        latest = df_c[df_c['year']==df_c['year'].max()].iloc[0]
        print(f'  {country:30s} {latest["renewable_pct"]:>9.2f}% '
              f'{latest["target_2030"]:>9.1f}% '
              f'{latest["gap_to_target"]:>9.1f}pp '
              f'{latest["pct_of_target_achieved"]:>10.1f}%')

FINAL QUALITY REPORT — GULF RENEWABLE ENERGY DATASET
  Total rows          : 96
  Total columns       : 13
  Countries           : 4
  Year span           : 2000 – 2023

  Missing Values in Master Dataset:
country_name              0
country_code              0
year                      0
renewable_pct             0
target_2030               0
gap_to_target             0
pct_of_target_achieved    0
is_saudi                  0
co2_per_capita            0
gdp_usd                   0
gdp_billion_usd           0
gdp_growth_pct            4
energy_use_per_capita     0

  Saudi Arabia — Key Statistics:
  Renewable % in 2000  : 0.00%
  Renewable % in 2023  : 0.07%
  Gap to 2030 target   : 49.9 percentage points
  2030 target          : 50.0%

  Latest Renewable % vs. 2030 Target (all countries):
  Country                          Latest %     Target        Gap   % Achieved
  ---------------------------------------------------------------------------
  Saudi Arabia                        0.07%

---
## Section 12: Save Cleaned Datasets

In [14]:
# ── Save master dataset (all countries) ───────────────────────────────────────
master_path = CLEANED_DIR + 'gulf_master.csv'
df_master.to_csv(master_path, index=False)
print(f'✅ Saved: {master_path}')

# ── Save Saudi Arabia slice ───────────────────────────────────────────────────
saudi_path = CLEANED_DIR + 'saudi_arabia.csv'
df_master[df_master['country_name'] == 'Saudi Arabia'].to_csv(saudi_path, index=False)
print(f'✅ Saved: {saudi_path}')

# ── Save Oman slice ───────────────────────────────────────────────────────────
oman_path = CLEANED_DIR + 'oman.csv'
df_master[df_master['country_name'] == 'Oman'].to_csv(oman_path, index=False)
print(f'✅ Saved: {oman_path}')

# ── Save renewable-only dataset (used most in Notebook 02) ────────────────────
renewable_path = CLEANED_DIR + 'gulf_renewable_clean.csv'
df_renewable.to_csv(renewable_path, index=False)
print(f'✅ Saved: {renewable_path}')

# ── List all saved files ──────────────────────────────────────────────────────
print()
print('Files in cleaned folder:')
for f in sorted(os.listdir(CLEANED_DIR)):
    size_kb = os.path.getsize(CLEANED_DIR + f) / 1024
    print(f'   ✅ {f:45s}  {size_kb:.1f} KB')

✅ Saved: ../data/cleaned/gulf_master.csv
✅ Saved: ../data/cleaned/saudi_arabia.csv
✅ Saved: ../data/cleaned/oman.csv
✅ Saved: ../data/cleaned/gulf_renewable_clean.csv

Files in cleaned folder:
   ✅ gulf_master.csv                                11.6 KB
   ✅ gulf_renewable_clean.csv                       5.3 KB
   ✅ oman.csv                                       2.9 KB
   ✅ saudi_arabia.csv                               3.1 KB


---
## Summary

| Step | Action | Status |
|------|--------|--------|
| 1 | Imported libraries | ✅ |
| 2 | Configured paths and constants | ✅ |
| 3 | Ran diagnostic on raw file structure | ✅ |
| 4 | Built smart World Bank loader function | ✅ |
| 5 | Loaded all 4 datasets | ✅ |
| 6 | Inspected shape, types, missing values | ✅ |
| 7 | Filled missing values via interpolation | ✅ |
| 8 | Validated all columns against logical bounds | ✅ |
| 9 | Created derived columns incl. gap-to-target | ✅ |
| 10 | Merged all datasets into master table | ✅ |
| 11 | Final quality report with Saudi Arabia summary | ✅ |
| 12 | Saved 4 clean CSV files to data/cleaned/ | ✅ |

---

**Next:** Open `02_exploratory_analysis.ipynb` to produce the 5 charts.